# Build Your AI Agent with MCPs using vLLM, Pydantic AI, and AMD MI300X GPU

Welcome to this hands-on workshop! Throughout this tutorial, we'll leverage AMD GPUs and **Model Context Protocol (MCP)** ,an open standard for exposing LLM tools via API, to deploy powerful language models like Qwen3. Key components:
- 🖥️ **vLLM** for GPU-optimized inference
- 🛠️ **Pydantic-AI** for agent/tool management
- 🔌 **MCP Servers** for pre-built tool integration

You'll learn how to set up your environment, deploy large language models like Qwen3, connect them to real-world tools using MCP, and build a conversational agent capable of reasoning and taking actions.

By the end of this workshop, you’ll have built an AI-powered Airbnb assistant agent—one that can find a place to stay based on your preferences like location, budget, and travel dates.

Let’s dive in!

## Table of Contents

- [Step 1: Launching vLLM Server on AMD GPUs](#step1)
- [Step 2: Installing Dependencies](#step2)
- [Step 3: Create a simple instance of Pydantic-AI Agent](#step3)
- [Step 4: Write a Date/Time Tool for Your Agent](#step4)
- [Step 5: Replace Your Date/time Tool with a MCP server](#step5)
- [Step 6: Turn your agent to an Airbnb finder](#step6)
- [Step 7: Challenge](#step7)

<a id="step1"></a>

## Step 1: Launch a vLLM Server

In this workshop we are going to use [vLLM](https://github.com/vllm-project/vllm) as our inference serving engine. vLLM provides many benefits such as fast model execution, extensive list of supported models, easy to use, and best of all it's open-source. 

### Deploy Qwen3-30B-A3B Model with vLLM




Time to start your vLLM server and creating an end-point for your LLM. Let's open a terminal using your Jupyter server. Then run the following command in this terminal to start the vLLM server:

```bash
VLLM_USE_TRITON_FLASH_ATTN=0 \
vllm serve Qwen/Qwen3-30B-A3B \
    --served-model-name Qwen3-30B-A3B \
    --api-key abc-123 \
    --port 8000 \
    --enable-auto-tool-choice \
    --tool-call-parser hermes \
    --trust-remote-code
```

Open another terminal and monitor the GPU utilization by running this command:

```bash
watch rocm-smi
```

Upon successful launch, your server should be accepting incoming traffic through an OpenAI-compatible API. Let's set some environment variables for our server so we can use throughout this tutorial:

In [1]:
import os

BASE_URL = f"http://localhost:8000/v1"

os.environ["BASE_URL"]    = BASE_URL
os.environ["OPENAI_API_KEY"] = "abc-123"   

print("Config set:", BASE_URL)

Config set: http://localhost:8000/v1


We can verify your model is available at the `BASE_URL` we just set by running the following command.

In [2]:
!curl http://localhost:8000/v1/models -H "Authorization: Bearer $OPENAI_API_KEY"

{"object":"list","data":[{"id":"Qwen3-30B-A3B","object":"model","created":1775735092,"owned_by":"vllm","root":"Qwen/Qwen3-30B-A3B","parent":null,"max_model_len":40960,"permission":[{"id":"modelperm-ac4def990eec2db4","object":"model_permission","created":1775735092,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

Congratulations, you now just launched a powerful server that can serve any incoming request and allowing you to build amazing applications. Wasn't that easy?🎉 

<a id="step2"></a>

## Step 2: Installing Dependencies

We are going to use `Pydantic AI`. Let's install the dependencies:

In [3]:
!pip install -q pydantic-ai-slim openai


<a id="step3"></a>

## Step 3: Create a simple instance of Pydantic-AI Agent

Let's start by creating a custom OpenAI Compatible endpoint for our agent. 


In [4]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

agent_model = OpenAIChatModel("Qwen3-30B-A3B", provider=provider)

Let's start by creating an instance the `Agent` class from `pydantic_ai`. 


In [5]:
from pydantic_ai import Agent

agent = Agent(
    model=agent_model
)

It's time to test the agent. `pydantic_ai` provides multiple ways to run `Agent`. You can learn more about it [here](https://ai.pydantic.dev/agents/#running-agents).

In this workshop, we are running in `async` mode. We are going to define a helper function that allows us to quickly test our agent throughout this workshop.

In [6]:
import asyncio
from pydantic_ai.mcp import MCPServerStdio
async def run_async(prompt: str) -> str:
    async with agent.run_mcp_servers():
        result = await agent.run(prompt)
        return result.output


Test the agent by calling this function.

In [7]:
await run_async("What is the capital of France?")

'\n\nThe capital of France is **Paris**. It is a major city in Europe, renowned for its cultural landmarks such as the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral. Paris serves as the political, economic, and cultural heart of the country.'

Great! now that we have the basics of creating an agent instance, and connecting it to the model we started serving with vLLM earlier.

<a id="step4"></a>

## Step 4: Write a Date/Time Tool for Your Agent

LLMs naturally rely on their training data to respond to your prompts. Therefore, the agent we just defined fails to answer a factual question that falls outside of it's training knowledge. Let's show this with an example:

In [8]:
await run_async("What’s the date today?")

"\n\nI don't have access to real-time information or current dates. My knowledge is based on data up to October 2023. For the current date, please check a reliable source or your device's calendar. Let me know if there's anything else I can help with! 😊"

It is no surprise that the model failed to answer this question. Now, it's time to power-up your LLM by providing `agent` a function that can get the current date. The process of an LLM triggering a function call is commonly referred to as `Tool Calling` or `Function Calling`. In this workshop we are going to take advantage of `pydantic-ai`'s agent `tools` to provide our agent appropriate tools. First, we need to define a custom tool. Below is how we can define a tool in this framework.

In [9]:
from datetime import datetime
from pydantic_ai import Tool          
@Tool
def get_current_date() -> str:
    """Return the current date/time as an ISO-formatted string."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


Next, we need to provide this tool to our Agent, as this will notify the LLM about the existence of such a tool we just definied. This is simply done by just providing the function signiture of the tool we just defined to our agent constructor. 

In [10]:
agent = Agent(
    model=agent_model,
    tools=[get_current_date],
    system_prompt = (
        "You have access to:\n"
        "   1. get_current_time(params: dict)\n"
        "Use this tool for date/time questions."
    )
)

Let's test the agent.

In [11]:
await run_async("What’s the date today?")

"\n\nToday's date is April 9, 2026, and the current time is 11:46:53."

Well done on building an agent with access to real-time data. 



<a id="step5"></a>

## Step 5: Replace Your Date/time Tool with a MCP server

Now that we learned how to create a custom tool and provide the agent access to this tool. Let's now explore a trendy topic of [Model Context Protocol](https://modelcontextprotocol.io/introduction). We are going to explore how we can replace our custom tool with a simple MCP server that can serve our agent and provide similar information.

**Why MCP?** MCP servers provide:
- ✅ Standardized API interfaces
- 🔄 Reusable across projects
- 📦 Pre-built functionality

Let's replace our custom time tool with an official MCP time server:

### Installing Time MCP Server

We are going to start by installing this MCP server:


In [12]:
!pip install -q mcp-server-time

Now let's define our time_server:

In [13]:
from pydantic_ai.mcp import MCPServerStdio

time_server = MCPServerStdio(
    "python",
    args=[
        "-m", "mcp_server_time",
        "--local-timezone=America/New_York",
    ],
)

Finally, let's modify our agent to remove our previously defined tool, and add this MCP server instead.

In [14]:
agent = Agent(
    model=agent_model,
    mcp_servers=[time_server],
    system_prompt = (
        "You are a helpful agent and you have access to this tool:\n"
        "   get_current_time(params: dict)\n"
        "When the user asks for the current date or time, call get_current_time.\n"
    )
)

Great, let's see if the agent can use the MCP to give us the correct time now.

In [15]:
await run_async("What’s the date today?")

"\n\nToday's date is Thursday, April 9, 2026."


Tadaa! Now you have officially used an MCP server to power-up your agent. In the next section we show how you can your turn many ideas into real working projects by using 100s of free or paid MCP servers available today.



<a id="step6"></a>

## Step 6: Turn your agent to an airbnb finder

As we experience in the last section, MCP servers are really easy to use and they provide a standard way of providing LLMs the tools we need. There are already thousands of MCP servers available for us to use. There are some MCP trackers that you can always use to find out about available servers. Here are some for your reference:
- https://github.com/modelcontextprotocol/servers
- https://mcp.so/

We are going to use npx to launch out next server. Therefore, let's install the required dependencies.

In [16]:
# Install Node.js 20 via NodeSource
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!apt install -y nodejs

2026-04-09 11:48:10 - Installing pre-requisites
Get:1 https://repo.radeon.com/amdgpu/30.30.1/ubuntu jammy InRelease [3185 B]
Get:2 https://repo.radeon.com/rocm/apt/7.2.1 jammy InRelease [2605 B]          
Get:3 https://repo.radeon.com/amdgpu/30.30.1/ubuntu jammy/main amd64 Packages [1332 B]3m
Get:4 https://repo.radeon.com/rocm/apt/7.2.1 jammy/main amd64 Packages [71.3 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy InRelease [270 kB]                
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [38.8 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]33m  
Get:11 http://archive.ubuntu.com/ubuntu jammy/restricted amd64 Packages [164 kB]
Get:12 http://archive.ubuntu.com/ubuntu 

Verify `npm` and `npx` installation:

In [17]:
!node -v && npm -v && npx --version

v20.20.2
10.8.2
10.8.2




In this part of the workshop we are going to build an agent that can help you browse available Airbnbs to book. We can now build on top of what we have so far and add an open-source Airbnb MCP server to our agent. To do so, let's start by defining our Airbnb server.

In [18]:
airbnb_server = MCPServerStdio(
    "npx", args=["-y", "@openbnb/mcp-server-airbnb", "--ignore-robots-txt"]
)

Let's update our agent.

In [19]:
system_prompt = """
You have access to three tools:
1. get_current_time(params: dict)
2. airbnb_search(params: dict)
3. airbnb_listing_details(params: dict)
When the user asks for listings, first call get_current_time, then airbnb_search, etc.
"""

agent = Agent(
    model=agent_model,
    mcp_servers=[time_server, airbnb_server],
    system_prompt=system_prompt,
)


Finally, let's try our agent and see if it can browse through Airbnb listings.

In [20]:
await run_async("Find a place to stay in Vancouver for next Sunday for 3 nights for 2 adults?")

'\n\nHere are some available Airbnb listings in Vancouver for your stay from April 12 to April 15, 2026 (3 nights):\n\n1. **Harbor View Condo**  \n   - Price: $347 for 3 nights  \n   - Ratings: 4.79 ⭐ (76 reviews)  \n   - Features: Pool, Gym  \n   - [View Listing](https://www.airbnb.com/rooms/1140605259279961390)\n\n2. **DT Pastoral @BC Place**  \n   - Price: $401 for 3 nights  \n   - Ratings: 4.9 ⭐ (99 reviews)  \n   - Features: Water view, Pool  \n   - [View Listing](https://www.airbnb.com/rooms/1027071906889057735)\n\n3. **Sky House - Panoramic Views**  \n   - Price: $452 for 3 nights  \n   - Ratings: 4.79 ⭐ (169 reviews)  \n   - Features: 3 bedrooms, 4 beds  \n   - [View Listing](https://www.airbnb.com/rooms/616990843631733627)\n\n4. **Spacious Downtown Loft**  \n   - Price: $473 for 3 nights  \n   - Ratings: 4.78 ⭐ (74 reviews)  \n   - Features: 1 bedroom, 2 beds  \n   - [View Listing](https://www.airbnb.com/rooms/33288404)\n\n5. **Modern 2BR in Lower Lonsdale**  \n   - Price: $37



<a id="step7"></a>

## Step 7: Challenge - Expand the Agent

**Task:** Add weather integration from any available MCP server you can find.

1. Launch weather MCP server
2. Add to agent's tools
3. Make agent suggest best travel dates based on weather

In [34]:
weather_server = MCPServerStdio(
    "npx",
    args=["-y", "@dangahagan/weather-mcp"]
)

In [35]:
system_prompt = """
You have access to the following tools:
1. get_current_time(params: dict)
2. airbnb_search(params: dict)
3. airbnb_listing_details(params: dict)
4. weather_forecast(params: dict)

When the user asks for travel:
1. Call get_current_time
2. Determine candidate travel dates
3. Call weather_forecast for those dates
4. Suggest best dates based on weather
5. Then call airbnb_search for the selected dates
6. Optionally call airbnb_listing_details
"""

In [ ]:
agent = Agent(
    model=agent_model,
    mcp_servers=[time_server, airbnb_server, weather_server],
    system_prompt=system_prompt,
)

In [33]:
for s in [time_server, airbnb_server, weather_server]:
    try:
        async with s:
            print("OK:", s)
    except Exception as e:
        print("FAILED:", s, e)

OK: MCPServerStdio(command='python', args=['-m', 'mcp_server_time', '--local-timezone=America/New_York'])
OK: MCPServerStdio(command='npx', args=['-y', '@openbnb/mcp-server-airbnb', '--ignore-robots-txt'])
FAILED: MCPServerStdio(command='npx', args=['-y', '@open-mcp/weather']) unhandled errors in a TaskGroup (1 sub-exception)


In [28]:
await run_async(
    "Find a place to stay in Vancouver for next Sunday for 3 nights for 2 adults?"
)

  + Exception Group Traceback (most recent call last):
  |   File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3745, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/tmp/ipykernel_82/2366767856.py", line 1, in <module>
  |     await run_async(
  |   File "/tmp/ipykernel_82/2171193897.py", line 4, in run_async
  |     async with agent.run_mcp_servers():
  |                ^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/usr/lib/python3.12/contextlib.py", line 210, in __aenter__
  |     return await anext(self.gen)
  |            ^^^^^^^^^^^^^^^^^^^^^
  |   File "/usr/local/lib/python3.12/dist-packages/pydantic_ai/agent/__init__.py", line 2597, in run_mcp_servers
  |     async with self:
  |                ^^^^
  |   File "/usr/local/lib/python3.12/dist-packages/pydantic_ai/agent/__init__.py", line 2474, in __aenter__
  |     await exit_stack.enter_async_context(toolset)
  |   File "/usr/lib/python3.12/contextlib.py", line 6

Happy coding! If you encounter issues or have questions, don’t hesitate to ask or raise an issue on our [Github page](https://github.com/ROCm/gpuaidev)!